# Calculating BII at the Ethnologue Polygon Level — additional years

Mirrors [`bii_ethnologue.ipynb`](bii_ethnologue.ipynb) (which already produces the year-2000 BII) but loops over the **post-2000 BII rasters** (years 2005, 2010, 2015, 2020 — all `v2-1-1`) and produces one mean-BII column per year.

Output: wide CSV `ethnologue_bii_byyear.csv` with columns `ID, area_km2, bii_2005, bii_2010, bii_2015, bii_2020`.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import rasterio
from rasterstats import zonal_stats

In [3]:
# Set base project path
base_path = Path("C:/Users/juami/Dropbox/RAships/2-Folklore-Nathan-Project/EA-Maps-Nathan-project/Measures_work")

poscol_path = base_path / "data" / "raw" / "ethnologue" / "ancestral_characteristics_database_language_level" / "Ethnologue_16_shapefile" / "langa_no_overlap_biggest_clean.shp"

maps_path = base_path / "maps" / "raw"
bii_dir   = maps_path / "BII"

# 2000 is already produced by bii_ethnologue.ipynb → skip it here
YEARS = [2005, 2010, 2015, 2020]
bii_files = {year: bii_dir / f"bii-{year}_v2-1-1.tif" for year in YEARS}

for year, fp in bii_files.items():
    print(f"{year}: {'OK' if fp.exists() else 'MISSING'} — {fp.name}")

2005: OK — bii-2005_v2-1-1.tif
2010: OK — bii-2010_v2-1-1.tif
2015: OK — bii-2015_v2-1-1.tif
2020: OK — bii-2020_v2-1-1.tif


In [4]:
# Load Ethnologue polygons
ethnologue = gpd.read_file(poscol_path)

# Compute polygon area in km² using equal-area projection (EPSG:6933)
ethnologue_proj = ethnologue.to_crs(epsg=6933)
ethnologue["area_km2"] = ethnologue_proj.geometry.area / 1e6

# Reproject to BII raster CRS (use first available raster as reference)
with rasterio.open(bii_files[YEARS[0]]) as ref:
    ref_crs = ref.crs
ethnologue = ethnologue.to_crs(ref_crs)

print(f"Number of features: {len(ethnologue)} | Raster CRS: {ref_crs}")

Number of features: 7087 | Raster CRS: EPSG:4326


In [5]:
# Loop over years and compute mean BII per polygon
for year, fp in bii_files.items():
    if not fp.exists():
        print(f"Skipping {year} — file missing")
        continue
    stats = zonal_stats(ethnologue, str(fp), stats=["mean"], geojson_out=False)
    ethnologue[f"bii_{year}"] = [s["mean"] for s in stats]
    print(f"Done year {year}")

Done year 2005
Done year 2010
Done year 2015
Done year 2020


In [6]:
# Build the output dataframe
year_cols = [f"bii_{y}" for y in YEARS if f"bii_{y}" in ethnologue.columns]
df_bii = ethnologue[["ID", "area_km2"] + year_cols].copy()

print(len(df_bii))
df_bii.head()

7087


,ID,area_km2,bii_2005,bii_2010,bii_2015,bii_2020
0,RUS-RUS,8.056159e+06,61.248978,60.906836,60.558717,60.280536
1,ENG-USA,7.104573e+06,45.061851,44.951382,44.775148,44.632869
2,POR-BRA,6.780013e+06,60.330522,60.237942,59.464734,59.676080
3,ENG-AUS,3.764472e+06,46.902565,46.415197,45.935438,46.753735
4,CMN-CHN,3.257609e+06,44.795061,44.868729,44.840654,45.162215


In [7]:
# Quick descriptive table — see the BII trajectory across years
df_bii[year_cols].describe()

,bii_2005,bii_2010,bii_2015,bii_2020
count,6384.000000,6384.000000,6384.000000,6384.000000
mean,59.151111,58.508813,57.752121,57.524384
std,21.800213,21.438850,21.093083,20.822083
min,12.925000,12.915000,12.815000,12.910000
25%,41.366126,41.157368,40.448794,40.526426
50%,57.397345,56.719083,55.958704,56.055765
75%,77.890385,76.662522,75.590652,74.744196
max,99.490000,99.476667,99.470000,99.453333


In [8]:
# Quick look at year-over-year deltas (e.g., biodiversity decline 2005 → 2020)
if "bii_2020" in df_bii.columns and "bii_2005" in df_bii.columns:
    df_bii["bii_delta_2005_2020"] = df_bii["bii_2020"] - df_bii["bii_2005"]
    print(df_bii[["bii_2005", "bii_2020", "bii_delta_2005_2020"]].describe())

          bii_2005     bii_2020  bii_delta_2005_2020
count  6384.000000  6384.000000          6384.000000
mean     59.151111    57.524384            -1.626727
std      21.800213    20.822083             2.945137
min      12.925000    12.910000           -32.435000
25%      41.366126    40.526426            -2.926488
50%      57.397345    56.055765            -1.111344
75%      77.890385    74.744196             0.070000
max      99.490000    99.453333            13.200000


In [9]:
# Export to CSV
out_path = bii_dir / "ethnologue_bii_byyear.csv"
df_bii.to_csv(out_path, index=False)
print("Exported", out_path)

Exported C:\Users\juami\Dropbox\RAships\2-Folklore-Nathan-Project\EA-Maps-Nathan-project\Measures_work\maps\raw\BII\ethnologue_bii_byyear.csv
